In [0]:
dim_customers = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_customers")
dim_invoices = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_invoices")
dim_products = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.dim_products")
fact_invoice_line_items = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_invoice_line_items")
fact_payments = spark.read.table("global_wholesale_distributor.curated_wholesale_distributor.fact_payments")

In [0]:
dim_customers.show()

In [0]:
display(fact_invoice_line_items)

In [0]:
display(fact_payments)

In [0]:
# fact_payments datacube

payment_cube = fact_payments.join(
    dim_invoices, fact_payments.invoice_id == dim_invoices.invoice_id, "inner"
).join(
    dim_customers, dim_invoices.customer_id == dim_customers.customer_id, "inner"
)

display(payment_cube)

In [0]:
# fact_invoice_line_items datacube

invoice_line_items_cube = fact_invoice_line_items.join(
    dim_invoices, fact_invoice_line_items.invoice_id == dim_invoices.invoice_id, "inner"
).join(
    dim_customers, dim_invoices.customer_id == dim_customers.customer_id, "inner"
).join(
    dim_products, fact_invoice_line_items.product_id == dim_products.product_id, "inner"
)

display(invoice_line_items_cube)

In [0]:
# Drop duplicate columns before saving
payment_cube_clean = payment_cube.drop(dim_invoices.invoice_id).drop(dim_invoices.customer_id)
invoice_line_items_cube_clean = invoice_line_items_cube.drop(dim_invoices.invoice_id).drop(dim_invoices.customer_id).drop(dim_products.product_id)

payment_cube_clean.write.mode("overwrite").saveAsTable("global_wholesale_distributor.datacubes_wholesale_distributor.payments_cube")
invoice_line_items_cube_clean.write.mode("overwrite").saveAsTable("global_wholesale_distributor.datacubes_wholesale_distributor.invoice_line_items_cube")